In [ ]:
# record_raw_5s_remote.py
import datetime as dt
import time

from maxlab.comm import ApiComm
from maxlab.system import DelaySamples

HOST = "132.77.68.106"
PORT = 7215

SAVE_DIR = "/home/mxwbio/Data/recordings"
WELLS = [0]           # for MaxTwo; harmless if you only use well 0
WELL_FOR_GROUP = 0    # group_define well index
GROUP_NAME = "all_channels"
DURATION_S = 5


def send_or_warn(send, cmd: str):
    """Send command; print warning (but don't crash) if it fails."""
    try:
        r = send(cmd)
        return r
    except Exception as e:
        print(f"WARNING: command failed: {cmd!r} -> {e}")
        return None

def send_with_retry(send, cmd, retries=10, delay=0.1):
    last = None
    for _ in range(retries):
        try:
            return send(cmd)
        except Exception as e:
            last = e
            time.sleep(delay)
    raise RuntimeError(f"Command failed after retries: {cmd!r}\nLast error: {last}")


def main():
    file_name = "raw_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    channels = list(range(1024))  # 0..1023

    print(f"Connecting to {HOST}:{PORT} ...")
    api = ApiComm(HOST, PORT)
    send = api.send

    try:
        # Basic connectivity check (mirrors what you already do)
        # DelaySamples(0) is a known ApiObject that can be sent.
        r = api.send("help")   # or another simple command
        if r is None:
            raise RuntimeError("No response from server.")

        # Offset and settle (your RemoteRecordingManager uses ~11s)
        print("Running offset compensation (system_offset) and waiting...")
        send_or_warn(send, "system_offset")
        time.sleep(11)

        # Open directory + start file
        print(f"Opening save directory: {SAVE_DIR}")
        send_or_warn(send, f"saving_open_dir {SAVE_DIR}")

        print(f"Starting file: {file_name}")
        send_or_warn(send, f"saving_start_file {file_name}")

        # --- Enable RAW traces: use new file format + define group ---
        # Docs: group_define is for new format (legacy_file_format=False) and must be
        # called after start_file and before start_recording; group_delete_all recommended. :contentReference[oaicite:3]{index=3}
        #
        # Server command name for "set_legacy_format" can vary by installation.
        # We therefore try a couple of common spellings and continue if unsupported.
        print("Setting new saving format (legacy_format=False) [best-effort] ...")
        send_or_warn(send, "saving_set_legacy_format 0")
        send_or_warn(send, "saving_set_legacy_file_format 0")

        print("Clearing any existing saving groups ...")
        #send_or_warn(send, "saving_group_delete_all")
        send_with_retry(send, "saving_group_delete_all", retries=20, delay=0.05)

        print(f"Defining group '{GROUP_NAME}' for {len(channels)} channels (0..1023) ...")
        # Tutorial equivalent: s.group_define(0, "all_channels", list(range(1024))) :contentReference[oaicite:4]{index=4}
        # Command payload: space-separated channels (most robust for text parsers).
        ch_payload = " ".join(map(str, channels))
        #send_or_warn(send, f"saving_group_define {WELL_FOR_GROUP} {GROUP_NAME} {ch_payload}")
        send_with_retry(send, f"saving_group_define {WELL_FOR_GROUP} {GROUP_NAME} {ch_payload}", retries=20, delay=0.05)

        # Start recording
        print(f"Starting recording (wells={WELLS}) ...")
        # Some servers accept wells argument; if yours doesn't, it will ignore or error harmlessly.
        # If it errors, delete the argument and use just "saving_start_recording".
        r = send_or_warn(send, f"saving_start_recording {' '.join(map(str, WELLS))}")
        if r is None:
            send_or_warn(send, "saving_start_recording")

        print(f"Recording for {DURATION_S} seconds ...")
        time.sleep(DURATION_S)

        # Stop recording + close file
        print("Stopping recording ...")
        send_or_warn(send, "saving_stop_recording")

        print("Stopping file ...")
        send_or_warn(send, "saving_stop_file")

        print("Done.")

    finally:
        try:
            api.shutdown()
        except Exception:
            pass


if __name__ == "__main__":
    main()


In [1]:
import datetime as dt
import time
import maxlab as mx

# Server-side save path (this path is on the MaxLab Linux machine)
SAVE_DIR = "/home/mxwbio/Data/recordings/Test"

DURATION_S = 5.0
WELL = 0
GROUP_NAME = "all_channels"
CHANNELS = list(range(1024))  # 0..1023

def main():
    # This MUST succeed through the SSH tunnel (localhost:7215 forwarded)
    mx.initialize()

    s = mx.Saving()
    s.open_directory(SAVE_DIR)

    file_name = "raw_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    s.start_file(file_name)

    # Raw traces require defining a group before start_recording
    # (exactly as in the official saving example)
    s.group_delete_all()
    s.group_define(WELL, GROUP_NAME, CHANNELS)

    s.start_recording([0])  # MaxOne: [0] is safe; some setups also accept no args
    time.sleep(DURATION_S)
    s.stop_recording()
    s.stop_file()

if __name__ == "__main__":
    main()
